# Demonstartor System Model 1
## Electronic Filter System

### Description

The electronic filter system is a simple example of an electronic circuit that consists of a voltage source that generates a noisy sinusoidal signal.
This signal is then passed through two low-pass filters and a band-pass filter to remove high-frequency noise and unwanted frequency components.

<img src="./img/ElectronicFilterSysetmCircuit.png" alt="Filter System" width="800"/>


Change path to access `SysSimX` module

In [15]:
import os
import sys
from pathlib import Path
repo_root = Path.cwd().parent.parent
sys.path.insert(0, str(repo_root))

Import functionality to create equation-based system model

In [16]:
from SysSimX.core.units import _ureg
from SysSimX.core.physical_value import PhysicalValue

from SysSimX.equationbased.package import Package
from SysSimX.equationbased.model import Model
from SysSimX.equationbased.interfaces import RealInput, RealOutput
from SysSimX.equationbased.variable import Variable
from SysSimX.equationbased.equation import Equation
from SysSimX.equationbased.utilities import der, sin, pi, time

Define the hierachy of the system model:

```
FilterSystem/                     (top-level package)
├─ package.mo
├─ package.order
├─ demo.mo                        (top-level demo)
│
├─ Components/                    (reusable electrical components)
│  ├─ package.mo
│  ├─ package.order
│  ├─ TwoPin.mo                   (partial base)
│  ├─ Resistor.mo                 (R)
│  ├─ Capacitor.mo                (C)
│  └─ Inductor.mo                 (L)
│
├─ Filters/                       (compositions built from Components)
│  ├─ package.mo
│  ├─ package.order
│  ├─ LowPassRC.mo                (subsystem)
│  └─ BandPassRLC.mo              (subsystem)
│
├─ Sources/                       
│  ├─ package.mo
│  ├─ package.order
│  └─ VoltageSource.mo                             
```

In [17]:
# Print current working directory
print(f"Current working directory: {Path.cwd()}")
!rm -rf FilterSystem

Current working directory: /home/flo/repos/SystemSimulation/demos/FilterSystem


Create the packages for the system model:

In [18]:
system_package_name = "FilterSystem"
system_path = Path.cwd() / system_package_name
system_package = Package(name=system_package_name, path=system_path)

/home/flo/repos/SystemSimulation/demos/FilterSystem/FilterSystem/package.mo created.


1. Create signal source model

In [19]:
# Instantiate the Model class
signal_source = Model("SignalSource")

# Define the package
signal_source.within = system_package.name

# Add outputs
output = RealOutput("Uin", "V")
signal_source.add_output(output)

# Define parameters
param_A1 = Variable("A1", "Real", "V", 60)
param_A2 = Variable("A2", "Real", "V", 5)
param_f1 = Variable("f1", "Real", "Hz", 0.159)
param_f2 = Variable("f2", "Real", "Hz", 7.96)

# Add parameters to the model
parameters = [param_A1, param_A2, param_f1, param_f2]
for param in parameters:
    signal_source.add_parameter(param)

# Define internal variables
internal_var_s1 = Variable("s1", "Real", "V")
internal_var_s2 = Variable("s2", "Real", "V")
signal_source.add_variable(internal_var_s1)
signal_source.add_variable(internal_var_s2)

# Define equations
eqn1_lhs = internal_var_s1
eqn1_rhs = param_A2 * sin(2 * pi * param_f1 * time)
eqn1 = Equation(eqn1_lhs, eqn1_rhs)
signal_source.add_equation(eqn1)

eqn2_lhs = internal_var_s2
eqn2_rhs = param_A1 * sin(2 * pi * param_f2 * time)
eqn2 = Equation(eqn2_lhs, eqn2_rhs)
signal_source.add_equation(eqn2)

signal_source.to_mo()
print(signal_source.modelica_code)

within FilterSystem;
model SignalSource
  Modelica.Blocks.Interfaces.RealOutput Uin(unit="V");
  Real A1(unit="V") = 60;
  Real A2(unit="V") = 5;
  Real f1(unit="Hz") = 0.159;
  Real f2(unit="Hz") = 7.96;
  Real s1(unit="V");
  Real s2(unit="V");
equation
  s1 = (A2 * sin((((2.0 * 3.141592653589793) * f1) * time)));
  s2 = (A1 * sin((((2.0 * 3.141592653589793) * f2) * time)));
end SignalSource;


In [23]:
signal_source.outputs["Uin"]

2. Create Low-Pass Filter model

In [20]:
# Instantiate the Model class
low_pass_filter = Model("LowPassFilter")

# Define the package
low_pass_filter.within = system_package.name

# Add inputs
U_in = RealInput("U_in", "V")
low_pass_filter.add_input(U_in)

# Add outputs
U_out = RealOutput("U_out", "V")
low_pass_filter.add_output(U_out)

# Define parameters
param_R = Variable("R", "Real", "Ohm", 1000)
param_C = Variable("C", "Real", "F", 1e-6)
param_U0 = Variable("U0", "Real", "V", 0)

# Add parameters to the model
parameters = [param_R, param_C, param_U0]
for param in parameters:
    low_pass_filter.add_parameter(param)

# Define initial equation
eqn_init_lhs = U_out
eqn_init_rhs = param_U0
eqn_init = Equation(eqn_init_lhs, eqn_init_rhs)
low_pass_filter.add_initial_equation(eqn_init)

# Define equations
eqn1_lhs = der(U_out)
eqn1_rhs = (U_in - U_out) / (param_R * param_C)
eqn1 = Equation(eqn1_lhs, eqn1_rhs)
low_pass_filter.add_equation(eqn1)

low_pass_filter.to_mo()
print(low_pass_filter.modelica_code)

within FilterSystem;
model LowPassFilter
  Modelica.Blocks.Interfaces.RealInput U_in(unit="V");
  Modelica.Blocks.Interfaces.RealOutput U_out(unit="V");
  Real R(unit="Ohm") = 1000;
  Real C(unit="F") = 1e-06;
  Real U0(unit="V") = 0;
initial equation
  U_out = U0;
equation
  der(U_out) = ((U_in - U_out) / (R * C));
end LowPassFilter;


3. Create Band-Pass Filter model

In [21]:
# Instantiate the Model class
band_pass_filter = Model("BandPassFilter")

# Define the package
band_pass_filter.within = system_package.name

# Add inputs
U_in = RealInput("U_in", "V")
band_pass_filter.add_input(U_in)

# Add outputs
I = RealOutput("I", "A")
IL = RealOutput("IL", "A")
band_pass_filter.add_output(I)
band_pass_filter.add_output(IL)

# Define parameters
param_R = Variable("R", "Real", "Ohm", 1000)
param_C = Variable("C", "Real", "F", 1e-6)
param_L = Variable("L", "Real", "H", 1e-2)
param_IL0 = Variable("IL0", "Real", "A", 0)

# Internal variable
internal_I_R = Variable("I_R", "Real", "A")
internal_I_C = Variable("I_C", "Real", "A")

# Add parameters to the model
parameters = [param_R, param_C, param_L, param_IL0, internal_I_R, internal_I_C]
for param in parameters:
    band_pass_filter.add_parameter(param)

# Define initial equation
eqn_init_lhs = IL
eqn_init_rhs = param_IL0
eqn_init = Equation(eqn_init_lhs, eqn_init_rhs)
band_pass_filter.add_initial_equation(eqn_init)

# Define equations
eqn1_lhs = internal_I_R
eqn1_rhs = U_in / param_R
eqn1 = Equation(eqn1_lhs, eqn1_rhs)
band_pass_filter.add_equation(eqn1)

eqn2_lhs = internal_I_C
eqn2_rhs = param_C *  der(U_in)
eqn2 = Equation(eqn2_lhs, eqn2_rhs)
band_pass_filter.add_equation(eqn2)

eqn3_lhs = der(IL)
eqn3_rhs = U_in / param_L
eqn3 = Equation(eqn3_lhs, eqn3_rhs)
band_pass_filter.add_equation(eqn3)

eqn4_lhs = I
eqn4_rhs = internal_I_R + internal_I_C + IL
eqn4 = Equation(eqn4_lhs, eqn4_rhs)
band_pass_filter.add_equation(eqn4)

band_pass_filter.to_mo()
print(band_pass_filter.modelica_code)

within FilterSystem;
model BandPassFilter
  Modelica.Blocks.Interfaces.RealInput U_in(unit="V");
  Modelica.Blocks.Interfaces.RealOutput I(unit="A");
  Modelica.Blocks.Interfaces.RealOutput IL(unit="A");
  Real R(unit="Ohm") = 1000;
  Real C(unit="F") = 1e-06;
  Real L(unit="H") = 0.01;
  Real IL0(unit="A") = 0;
  Real I_R(unit="A");
  Real I_C(unit="A");
initial equation
  IL = IL0;
equation
  I_R = (U_in / R);
  I_C = (C * der(U_in));
  der(IL) = (U_in / L);
  I = ((I_R + I_C) + IL);
end BandPassFilter;


4. Create demo model to connect components

In [ ]:
# Create demo model to connect components
demo_model = Model("DemoModel")
demo_model.within = system_package.name

# Add components as variables
cmp_signal_source = Variable("src", signal_source.name, "", None)
demo_model.add_component(cmp_signal_source)

cmp_lpf1 = Variable("lpf1", low_pass_filter.name, "", None)
demo_model.add_component(cmp_lpf1)

cmp_lpf2 = Variable("lpf2", low_pass_filter.name, "", None)
demo_model.add_component(cmp_lpf2)

cmp_bpf = Variable("bpf", band_pass_filter.name, "", None)
demo_model.add_component(cmp_bpf)

# Add variables for outputs
src_voltage = Variable("src_voltage", "Real", "V"); demo_model.add_variable(src_voltage)
lpf1_voltage = Variable("lpf1_voltage", "Real", "V"); demo_model.add_variable(lpf1_voltage)
lpf2_voltage = Variable("lpf2_voltage", "Real", "V"); demo_model.add_variable(lpf2_voltage)
bpf_current = Variable("bpf_current", "Real", "A"); demo_model.add_variable(bpf_current)
bpf_IL = Variable("bpf_IL", "Real", "A"); demo_model.add_variable(bpf_IL)

# Define connecting equations
connection1 = Connection(signal_source.outputs.)

demo_model.to_mo()
print(demo_model.modelica_code)

within FilterSystem;
model DemoModel
  SignalSource src();
  LowPassFilter lpf1();
  LowPassFilter lpf2();
  BandPassFilter bpf();
  Real src_voltage(unit="V");
  Real lpf1_voltage(unit="V");
  Real lpf2_voltage(unit="V");
  Real bpf_current(unit="A");
  Real bpf_IL(unit="A");
end DemoModel;
